In [4]:
import pandas as pd

df = pd.read_excel('../data/raw/main.xlsx')

#پاکسازی دیتای نمره و تبدیل نمره
def Score_conversion(x):
    try:
        return float(x)
    except:
        return None

df['نمره_عددی'] = df['نمره'].apply(Score_conversion)
print("ready")

ready


In [5]:
profile = df.groupby('شماره‌دانشجويي').agg(
    میانگین_نمره = ('نمره_عددی' , 'mean'),
    تعداد_درس = ('نمره_عددی' , "count") , 
    تعداد_مردودی = ('نمره_عددی' , lambda x: (x < 10).sum()),
    تعداد_فاقد_برگه = ('نمره', lambda x:(x == 'فاقد برگه').sum()),

).reset_index()

In [6]:
print(profile.shape)
profile.head(10)

(7769, 5)


,شماره‌دانشجويي,میانگین_نمره,تعداد_درس,تعداد_مردودی,تعداد_فاقد_برگه
0,110095300001,NaN,0,0,6
1,110095300003,NaN,0,0,7
2,110095300004,11.882857,35,9,13
3,110095300005,14.644737,38,1,0
4,110095300007,11.455000,50,14,1
5,110095300009,13.250000,9,1,4
6,110095300010,9.221154,26,11,2
7,110095300011,15.444444,9,1,15
8,110095300012,13.610000,25,4,3
9,110095300014,NaN,0,0,7


In [7]:
# اگه وضعیت توی لیست خطر بود True بذار
risk_status = [
    'حذف ترم باسنوات',
    'حذف ترم بدون سنوات',
    'محروم از تحصيل با احتساب در سنوات',
    'تعليق ترم با احتساب در سنوات',
    'تعليق ترم بدون احتساب در سنوات'
]

df['خطرناک'] = df['وضعيت‌ترم‌دانشجو'].isin(risk_status)


In [8]:
print(df.columns.tolist())

['کدترم امتحان', 'شماره\u200cدانشجويي', 'کد\u200cترم\u200cورود دانشجو', 'درس', 'کددرس', 'نمره', 'نوع\u200cنمره', 'کد\u200cگروه\u200cدرسي', 'کدملي\u200cمدرس', 'نام\u200cمدرس', 'وضعيت\u200cترم\u200cدانشجو', 'شناسه\u200cدانشكده', 'نمره_عددی', 'خطرناک']


In [9]:
student_risk = df.groupby('شماره‌دانشجويي')['خطرناک'].max()

In [10]:
profile['در_معرض_خطر'] = profile['شماره‌دانشجويي'].map(student_risk).astype(int)

profile['در_معرض_خطر'].value_counts()

در_معرض_خطر
0    7564
1     205
Name: count, dtype: int64

In [11]:
profile['در_معرض_خطر'] = (
    (profile['در_معرض_خطر'] == 1) |
    (profile['میانگین_نمره'] < 12) |
    (profile['تعداد_مردودی'] > 3)
).astype(int)

profile['در_معرض_خطر'].value_counts()

در_معرض_خطر
0    5569
1    2200
Name: count, dtype: int64

In [12]:
df.columns.tolist()

['کدترم امتحان',
 'شماره\u200cدانشجويي',
 'کد\u200cترم\u200cورود دانشجو',
 'درس',
 'کددرس',
 'نمره',
 'نوع\u200cنمره',
 'کد\u200cگروه\u200cدرسي',
 'کدملي\u200cمدرس',
 'نام\u200cمدرس',
 'وضعيت\u200cترم\u200cدانشجو',
 'شناسه\u200cدانشكده',
 'نمره_عددی',
 'خطرناک']

In [13]:
uni = df.groupby('شماره‌دانشجويي')['شناسه‌دانشكده'].first()
profile['دانشکده'] = profile['شماره‌دانشجويي'].map(uni)
profile['دانشکده'].value_counts()

دانشکده
دختران قزوین    4125
پسران قزوین     3644
Name: count, dtype: int64

In [14]:
# درصد در معرض خطر در هر دانشکده
profile.groupby('دانشکده')['در_معرض_خطر'].mean() * 100


دانشکده
دختران قزوین    20.509091
پسران قزوین     37.156970
Name: در_معرض_خطر, dtype: float64

In [15]:
# میانگین نمره هر دانشکده
profile.groupby('دانشکده')['میانگین_نمره'].mean()

دانشکده
دختران قزوین    15.576494
پسران قزوین     13.611084
Name: میانگین_نمره, dtype: float64

In [16]:
#تعداد دانشحویان دانشکده دختران و پسران
profile.groupby('دانشکده')['در_معرض_خطر'].count()

دانشکده
دختران قزوین    4125
پسران قزوین     3644
Name: در_معرض_خطر, dtype: int64

In [17]:
# ورودهای در معرض خط
term_vorod = df.groupby('شماره‌دانشجويي')['کد‌ترم‌ورود دانشجو'].first()
profile['ترم_ورود'] = profile['شماره‌دانشجويي'].map(term_vorod)
profile.groupby('ترم_ورود')['در_معرض_خطر'].mean() * 100

ترم_ورود
1      31.722881
2      40.377804
11     40.549102
12     39.530988
21     28.380024
22     26.966292
31      7.544582
951    75.000000
961    33.333333
962    20.000000
971    21.739130
972    30.769231
981    11.637931
982    14.444444
991    17.673378
992    31.985940
Name: در_معرض_خطر, dtype: float64

In [18]:
#تعداد دانشجویان هر ترم که در معرض خط هستن
profile.groupby('ترم_ورود')['در_معرض_خطر'].count()

ترم_ورود
1      1097
2       847
11      947
12      597
21      821
22      623
31      729
951       4
961       3
962      10
971      23
972      13
981     232
982     360
991     894
992     569
Name: در_معرض_خطر, dtype: int64

In [19]:
#حذف ورودی های نامعتبر
vorod_motaber = profile['ترم_ورود'].value_counts()
vorod_motaber = vorod_motaber[vorod_motaber >= 100].index

filter_profile = profile[profile['ترم_ورود'].isin(vorod_motaber)]
print(filter_profile.shape)

(7716, 8)


In [23]:
filter_profile.to_excel('../data/processed/پروفایل_دانشجو.xlsx', index=False)

In [24]:
filter_profile.columns.to_list()

['شماره\u200cدانشجويي',
 'میانگین_نمره',
 'تعداد_درس',
 'تعداد_مردودی',
 'تعداد_فاقد_برگه',
 'در_معرض_خطر',
 'دانشکده',
 'ترم_ورود']